In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
df = pd.read_csv('../data/raw/ico/prices-paid-to-growers.csv')

In [3]:
df

,prices_paid_to_growers,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
0,Colombia,1.534724,1.481790,1.204656,1.106477,1.898327,2.199185,2.065245,2.939673,2.253433,...,3.067442,3.985616,5.290868,3.679737,2.514539,3.525662,2.635854,2.727530,2.776185,2.505914
1,Dominican Republic,1.458168,1.382845,1.027841,1.172704,2.478234,2.412000,1.935342,3.673587,2.102616,...,2.340731,3.071276,4.217812,3.262839,3.112084,4.052172,3.775011,3.911395,3.826974,3.391053
2,El Salvador,1.116194,0.983322,0.682322,0.780397,2.191177,2.218826,1.656073,2.762552,1.877384,...,1.748205,2.425620,4.122256,2.651658,2.103276,2.582929,1.930439,1.976442,1.934159,1.645300
3,Guatemala,1.204956,1.270086,0.888099,0.914552,1.662711,2.192936,1.849170,2.146819,2.261521,...,2.420375,3.195830,4.689066,3.664007,2.804848,3.373810,3.073230,2.990444,2.960790,2.697022
4,Honduras,1.111470,1.238947,0.886057,0.828746,1.800576,2.263781,1.493616,2.580166,2.058671,...,1.848015,2.762887,4.415430,3.115841,2.247159,2.625614,2.449565,2.056691,2.003671,1.855291
5,India,1.473558,1.358371,1.191159,1.278669,1.730810,1.870104,1.728881,2.873698,2.195885,...,3.021792,3.335051,5.043230,3.575634,2.584011,3.877464,3.467918,2.918461,2.996744,2.430896
6,Uganda,0.337598,0.654322,0.441397,0.552298,1.666651,1.506821,1.169927,1.772936,1.738302,...,1.336532,1.925611,3.245623,2.068040,1.545905,2.289245,1.794042,1.569419,1.699698,1.506691
7,Brazil,1.199223,0.971150,0.997768,1.167263,2.529110,2.550172,2.126834,3.209923,2.323380,...,2.225150,2.957956,4.950552,3.361764,2.254700,2.973442,2.306547,2.380366,2.438757,2.003980
8,Ethiopia,1.348565,1.505322,1.351128,1.362442,2.418234,2.539011,1.495287,2.010060,2.027901,...,1.701150,1.886530,3.211088,2.269088,1.618110,1.837430,1.678360,1.927316,1.799278,1.560234
9,Brazil,0.769285,0.606947,0.609155,0.784638,1.885786,2.152316,1.540377,1.718671,1.716276,...,1.620826,1.638355,2.324366,2.304737,1.882698,1.791693,1.623110,2.048073,2.168022,1.488327


### Remove non-arabica rows

In [4]:
df = df.drop([9,10,11,12])

In [5]:
df_2 = pd.read_csv('../data/raw/ico/indicator-prices.csv')

In [6]:
df_2.head()

,months,ICO composite indicator,Colombian Milds,Other Milds,Brazilian Naturals,Robustas
0,01/1990,1.385210,1.811700,1.678146,1.553201,1.092053
1,02/1990,1.479249,2.020971,1.853201,1.712804,1.105077
2,03/1990,1.661148,2.279029,2.091170,1.902208,1.230905
3,04/1990,1.663135,2.247020,2.090728,1.930464,1.235099
4,05/1990,1.618102,2.188521,2.052318,1.905298,1.183664


### Parse the months column and retrieve only the arabic columns

In [7]:
ind = df_2.copy()
ind['year'] = pd.to_datetime(df_2['months'], format='%m/%Y').dt.year
arabica = ['Colombian Milds', 'Other Milds', 'Brazilian Naturals']
ind_long = (ind.groupby('year')[arabica].mean().reset_index().melt(
    id_vars='year', 
    var_name='group', 
    value_name='indicator_price'
))
ind_long.head()

,year,group,indicator_price
0,1990,Colombian Milds,2.130979
1,1991,Colombian Milds,1.981512
2,1992,Colombian Milds,1.500331
3,1993,Colombian Milds,1.673013
4,1994,Colombian Milds,3.471762


### Add indicator labels into grower data

In [8]:
group_map = {0: 'Colombian Milds',
             1: 'Other Milds', 2: 'Other Milds', 3: 'Other Milds',
             4: 'Other Milds', 5: 'Other Milds', 6: 'Other Milds',
             7: 'Brazilian Naturals', 8: 'Brazilian Naturals'}
gp = df.copy()
gp['group'] = gp.index.map(group_map)
gp_long = (gp.rename(columns={'prices_paid_to_growers': 'country'})
             .melt(id_vars=['country', 'group'], var_name='year', value_name='grower_price'))
gp_long['year'] = gp_long['year'].astype(int)
gp_long.head()

,country,group,year,grower_price
0,Colombia,Colombian Milds,1990,1.534724
1,Dominican Republic,Other Milds,1990,1.458168
2,El Salvador,Other Milds,1990,1.116194
3,Guatemala,Other Milds,1990,1.204956
4,Honduras,Other Milds,1990,1.111470


### Combine the two dataframes

In [9]:
merged = gp_long.merge(ind_long, on=['group', 'year'], how='left')
merged['grower_share'] = merged['grower_price'] / merged['indicator_price']
merged[['grower_price', 'indicator_price']] *= 1000
merged

,country,group,year,grower_price,indicator_price,grower_share
0,Colombia,Colombian Milds,1990,1534.724062,2130.978661,0.720197
1,Dominican Republic,Other Milds,1990,1458.167770,1974.760854,0.738402
2,El Salvador,Other Milds,1990,1116.194260,1974.760854,0.565230
3,Guatemala,Other Milds,1990,1204.955850,1974.760854,0.610178
4,Honduras,Other Milds,1990,1111.470199,1974.760854,0.562838
...,...,...,...,...,...,...
256,Honduras,Other Milds,2018,1855.291391,2929.801325,0.633248
257,India,Other Milds,2018,2430.896247,2929.801325,0.829714
258,Uganda,Other Milds,2018,1506.690949,2929.801325,0.514264
259,Brazil,Brazilian Naturals,2018,2003.980132,2508.388521,0.798911


### Adjust prices for inflation

In [10]:
cpi = pd.DataFrame({
    'year': list(range(1990, 2019)),
    'cpi': [130.7, 136.2, 140.3, 144.5, 148.2, 152.4, 156.9, 160.5, 163.0, 166.6,
            172.2, 177.1, 179.9, 184.0, 188.9, 195.3, 201.6, 207.3, 215.3, 214.5,
            218.1, 224.9, 229.6, 233.0, 236.7, 237.0, 240.0, 245.1, 251.1]})

In [11]:
merged = merged.merge(cpi, on='year', how='left')
cpi_2018 = cpi.loc[cpi['year'] == 2018, 'cpi'].iloc[0]
merged['deflator'] = cpi_2018 / merged['cpi']

merged = merged.rename(columns={'grower_price': 'grower_price_nominal',
                                'indicator_price': 'indicator_price_nominal'})
merged['grower_price_real_2018'] = merged['grower_price_nominal'] * merged['deflator']
merged['indicator_price_real_2018'] = merged['indicator_price_nominal'] * merged['deflator']

In [12]:
merged.head()

,country,group,year,grower_price_nominal,indicator_price_nominal,grower_share,cpi,deflator,grower_price_real_2018,indicator_price_real_2018
0,Colombia,Colombian Milds,1990,1534.724062,2130.978661,0.720197,130.7,1.921194,2948.502004,4094.022507
1,Dominican Republic,Other Milds,1990,1458.167770,1974.760854,0.738402,130.7,1.921194,2801.422549,3793.897860
2,El Salvador,Other Milds,1990,1116.194260,1974.760854,0.565230,130.7,1.921194,2144.425240,3793.897860
3,Guatemala,Other Milds,1990,1204.955850,1974.760854,0.610178,130.7,1.921194,2314.953435,3793.897860
4,Honduras,Other Milds,1990,1111.470199,1974.760854,0.562838,130.7,1.921194,2135.349402,3793.897860


In [14]:
merged.to_csv('../data/processed/ico_2018_cleaned.csv', index=False)